In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split



   user_id  movie_id  rating  timestamp
0      196       242       3  881250949
1      186       302       3  891717742
2       22       377       1  878887116
3      244        51       2  880606923
4      166       346       1  886397596
(100000, 4)


In [ ]:
ratings = pd.read_csv(
    'ml-100k/u.data',
    sep='\t',
    names=['user_id', 'movie_id', 'rating', 'timestamp']
)

# Remap IDs to start at 0
user_map  = {u: i for i, u in enumerate(ratings['user_id'].unique())}
movie_map = {m: i for i, m in enumerate(ratings['movie_id'].unique())}

ratings['user_idx']  = ratings['user_id'].map(user_map)
ratings['movie_idx'] = ratings['movie_id'].map(movie_map)

n_users  = len(user_map)
n_movies = len(movie_map)

print(f"Users: {n_users}, Movies: {n_movies}")

# Normalize ratings to 0-1 range
ratings['rating_norm'] = (ratings['rating'] - 1) / 4.0

# Train/test split
train_df, test_df = train_test_split(ratings, test_size=0.2, random_state=42)

Users: 943, Movies: 1682


In [3]:
class MovieDataset(Dataset):
    def __init__(self, df):
        self.users  = torch.tensor(df['user_idx'].values,   dtype=torch.long)
        self.movies = torch.tensor(df['movie_idx'].values,  dtype=torch.long)
        self.ratings = torch.tensor(df['rating_norm'].values, dtype=torch.float32)

    def __len__(self):
        return len(self.ratings)

    def __getitem__(self, idx):
        return self.users[idx], self.movies[idx], self.ratings[idx]


train_dataset = MovieDataset(train_df)
test_dataset  = MovieDataset(test_df)

train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=256, shuffle=False)

In [4]:
class MatrixFactorization(nn.Module):
    def __init__(self, n_users, n_movies, n_factors=64, dropout=0.1):
        super().__init__()
        
        # Embedding layers — these are the "latent factors"
        self.user_embeddings  = nn.Embedding(n_users,  n_factors)
        self.movie_embeddings = nn.Embedding(n_movies, n_factors)
        
        # Bias terms for each user and movie
        self.user_bias  = nn.Embedding(n_users,  1)
        self.movie_bias = nn.Embedding(n_movies, 1)
        
        self.dropout = nn.Dropout(dropout)
        
        # Initialize weights — important for stable training
        nn.init.xavier_uniform_(self.user_embeddings.weight)
        nn.init.xavier_uniform_(self.movie_embeddings.weight)
        nn.init.zeros_(self.user_bias.weight)
        nn.init.zeros_(self.movie_bias.weight)

    def forward(self, user, movie):
        user_emb  = self.dropout(self.user_embeddings(user))
        movie_emb = self.dropout(self.movie_embeddings(movie))
        
        # Dot product of user and movie factors
        dot = (user_emb * movie_emb).sum(dim=1)
        
        # Add biases
        bias = self.user_bias(user).squeeze() + self.movie_bias(movie).squeeze()
        
        # Sigmoid to keep output between 0 and 1 (matches normalized ratings)
        return torch.sigmoid(dot + bias)


model = MatrixFactorization(n_users, n_movies, n_factors=64)
print(model)

MatrixFactorization(
  (user_embeddings): Embedding(943, 64)
  (movie_embeddings): Embedding(1682, 64)
  (user_bias): Embedding(943, 1)
  (movie_bias): Embedding(1682, 1)
  (dropout): Dropout(p=0.1, inplace=False)
)
